#### Ispezione cammini BGP

Dovrebbe essere del tipo: routeviews/routeviews|5 1239|6113|8063|3464 199.88.21.0/24 i 144.228.240.93

In [91]:
import bz2

with bz2.open("/code/ADSproject/data/20110501.all-paths.bz2", "rt") as f:
    for i, riga in enumerate(f):
        print(riga)
        if i > 5:  # guarda solo le prime righe
            break

routeviews/isc|5 4436|6762|21826 200.82.128.0/24 i 198.32.176.13

routeviews/isc|5 6939|15290|2671|2669 198.103.53.0/24 i 198.32.176.20

routeviews/isc|5 4826|6939|3549|7922|20214 70.89.88.0/24 i 198.32.176.134

routeviews/isc|5 14361|15290|23373 216.9.51.0/24 i 198.32.176.10

routeviews/isc|5 14361|4766|1237 203.254.160.0/21 i 198.32.176.10

routeviews/isc|5 4589|3356|12956|3816|8163 190.182.7.0/24 i 198.32.176.74

routeviews/isc|5 7575|6939|22637 67.208.228.0/24 i 198.32.176.177



#### Ispezione nodi

Dovrebbe essere del tipo: AS1|AS2|relationship (relationship non ci interessa)

In [92]:
import bz2

with bz2.open("/code/ADSproject/data/20110501.as-rel.txt.bz2", "rt") as f:
    for i, riga in enumerate(f):
        print(riga)
        if i > 5:  # guarda solo le prime righe
            break

# source:topology|BGP|20110501|ripe|rrc00

# source:topology|BGP|20110502|ripe|rrc00

# source:topology|BGP|20110503|ripe|rrc00

# source:topology|BGP|20110504|ripe|rrc00

# source:topology|BGP|20110505|ripe|rrc00

# source:topology|BGP|20110501|ripe|rrc01

# source:topology|BGP|20110502|ripe|rrc01



In [93]:
import bz2

with bz2.open("/code/ADSproject/data/20110501.as-rel.txt.bz2", "rt") as f:
    for i, riga in enumerate(f):
        if riga.startswith("#"):
            continue  # salta i commenti
        print(riga.strip())
        break  # guarda solo la prima riga utile

1|2|-1


#### Prova estrazione dati (parsing dei cammini dal file all paths)

In [94]:
def leggi_cammini(filepath, max_righe=None):
    cammini = []
    contatore = 0
    with bz2.open(filepath, "rt") as f:
        for riga in f:
            if max_righe and contatore >= max_righe:
                break
            if riga.startswith("#"):
                continue
            parti = riga.strip().split()
            print(parti)
            cammino = []
            for p in parti[1:]:
                if "/" in p or "." in p or ":" in p:  
                    break
                nodi = p.split("|")
                print(nodi)
                cammino.extend(nodi)
            if len(cammino) > 1:  ## aggiungi solo cammini con almeno due nodi 
                cammini.append(cammino)
                contatore += 1
    return cammini

In [95]:
# test
cammini = leggi_cammini("/code/ADSproject/data/20110501.all-paths.bz2", max_righe=10)  # leggi solo le prime 10 righe per test
print(cammini[:5])  # stampa i primi 5 cammini


['routeviews/isc|5', '4436|6762|21826', '200.82.128.0/24', 'i', '198.32.176.13']
['4436', '6762', '21826']
['routeviews/isc|5', '6939|15290|2671|2669', '198.103.53.0/24', 'i', '198.32.176.20']
['6939', '15290', '2671', '2669']
['routeviews/isc|5', '4826|6939|3549|7922|20214', '70.89.88.0/24', 'i', '198.32.176.134']
['4826', '6939', '3549', '7922', '20214']
['routeviews/isc|5', '14361|15290|23373', '216.9.51.0/24', 'i', '198.32.176.10']
['14361', '15290', '23373']
['routeviews/isc|5', '14361|4766|1237', '203.254.160.0/21', 'i', '198.32.176.10']
['14361', '4766', '1237']
['routeviews/isc|5', '4589|3356|12956|3816|8163', '190.182.7.0/24', 'i', '198.32.176.74']
['4589', '3356', '12956', '3816', '8163']
['routeviews/isc|5', '7575|6939|22637', '67.208.228.0/24', 'i', '198.32.176.177']
['7575', '6939', '22637']
['routeviews/isc|5', '4436|701|702', '213.116.184.0/23', 'i', '198.32.176.13']
['4436', '701', '702']
['routeviews/isc|5', '4436|2914|7018|12217', '153.2.77.0/24', 'i', '198.32.176.13'

#### Prova costruzione frequenze a partire dai cammini estratti 
(lista di liste del tipo [['4436', '6762', '21826'], ['6939', '15290', '2671', '2669']])

In [96]:
from collections import defaultdict

frequenze = defaultdict(int)

for cammino in cammini:
        for i in range(len(cammino) - 1):
            u = cammino[i]
            v = cammino[i + 1]
            if u == v:  # ignora self-loop
                continue
            # stampa l'arco
            print(f"Arco: {u} -> {v}")
            u_int, v_int = int(u), int(v)
            arco = (min(u_int, v_int), max(u_int, v_int)) ## per non avere archi duplicati del tipo (1,2) e (2,1)
            frequenze[arco] = frequenze.get(arco, 0) + 1
            print(f"Frequenza arco {u} -> {v}: {frequenze[arco]}")

Arco: 4436 -> 6762
Frequenza arco 4436 -> 6762: 1
Arco: 6762 -> 21826
Frequenza arco 6762 -> 21826: 1
Arco: 6939 -> 15290
Frequenza arco 6939 -> 15290: 1
Arco: 15290 -> 2671
Frequenza arco 15290 -> 2671: 1
Arco: 2671 -> 2669
Frequenza arco 2671 -> 2669: 1
Arco: 4826 -> 6939
Frequenza arco 4826 -> 6939: 1
Arco: 6939 -> 3549
Frequenza arco 6939 -> 3549: 1
Arco: 3549 -> 7922
Frequenza arco 3549 -> 7922: 1
Arco: 7922 -> 20214
Frequenza arco 7922 -> 20214: 1
Arco: 14361 -> 15290
Frequenza arco 14361 -> 15290: 1
Arco: 15290 -> 23373
Frequenza arco 15290 -> 23373: 1
Arco: 14361 -> 4766
Frequenza arco 14361 -> 4766: 1
Arco: 4766 -> 1237
Frequenza arco 4766 -> 1237: 1
Arco: 4589 -> 3356
Frequenza arco 4589 -> 3356: 1
Arco: 3356 -> 12956
Frequenza arco 3356 -> 12956: 1
Arco: 12956 -> 3816
Frequenza arco 12956 -> 3816: 1
Arco: 3816 -> 8163
Frequenza arco 3816 -> 8163: 1
Arco: 7575 -> 6939
Frequenza arco 7575 -> 6939: 1
Arco: 6939 -> 22637
Frequenza arco 6939 -> 22637: 1
Arco: 4436 -> 701
Frequenz

In [97]:
frequenze

defaultdict(int,
            {(4436, 6762): 1,
             (6762, 21826): 1,
             (6939, 15290): 1,
             (2671, 15290): 1,
             (2669, 2671): 1,
             (4826, 6939): 1,
             (3549, 6939): 1,
             (3549, 7922): 1,
             (7922, 20214): 1,
             (14361, 15290): 1,
             (15290, 23373): 1,
             (4766, 14361): 1,
             (1237, 4766): 1,
             (3356, 4589): 1,
             (3356, 12956): 1,
             (3816, 12956): 1,
             (3816, 8163): 1,
             (6939, 7575): 1,
             (6939, 22637): 1,
             (701, 4436): 1,
             (701, 702): 1,
             (2914, 4436): 1,
             (2914, 7018): 1,
             (7018, 12217): 1,
             (1273, 7575): 1,
             (1273, 20485): 1,
             (20485, 21127): 1})

In [98]:
for arco, freq in frequenze.items():
        print(f"Arco: {arco}, Frequenza: {freq}")

Arco: (4436, 6762), Frequenza: 1
Arco: (6762, 21826), Frequenza: 1
Arco: (6939, 15290), Frequenza: 1
Arco: (2671, 15290), Frequenza: 1
Arco: (2669, 2671), Frequenza: 1
Arco: (4826, 6939), Frequenza: 1
Arco: (3549, 6939), Frequenza: 1
Arco: (3549, 7922), Frequenza: 1
Arco: (7922, 20214), Frequenza: 1
Arco: (14361, 15290), Frequenza: 1
Arco: (15290, 23373), Frequenza: 1
Arco: (4766, 14361), Frequenza: 1
Arco: (1237, 4766), Frequenza: 1
Arco: (3356, 4589), Frequenza: 1
Arco: (3356, 12956), Frequenza: 1
Arco: (3816, 12956), Frequenza: 1
Arco: (3816, 8163), Frequenza: 1
Arco: (6939, 7575), Frequenza: 1
Arco: (6939, 22637), Frequenza: 1
Arco: (701, 4436), Frequenza: 1
Arco: (701, 702), Frequenza: 1
Arco: (2914, 4436), Frequenza: 1
Arco: (2914, 7018), Frequenza: 1
Arco: (7018, 12217), Frequenza: 1
Arco: (1273, 7575), Frequenza: 1
Arco: (1273, 20485), Frequenza: 1
Arco: (20485, 21127), Frequenza: 1


In [99]:
frequenze

defaultdict(int,
            {(4436, 6762): 1,
             (6762, 21826): 1,
             (6939, 15290): 1,
             (2671, 15290): 1,
             (2669, 2671): 1,
             (4826, 6939): 1,
             (3549, 6939): 1,
             (3549, 7922): 1,
             (7922, 20214): 1,
             (14361, 15290): 1,
             (15290, 23373): 1,
             (4766, 14361): 1,
             (1237, 4766): 1,
             (3356, 4589): 1,
             (3356, 12956): 1,
             (3816, 12956): 1,
             (3816, 8163): 1,
             (6939, 7575): 1,
             (6939, 22637): 1,
             (701, 4436): 1,
             (701, 702): 1,
             (2914, 4436): 1,
             (2914, 7018): 1,
             (7018, 12217): 1,
             (1273, 7575): 1,
             (1273, 20485): 1,
             (20485, 21127): 1})

In [100]:
dizio = defaultdict(dict)

In [101]:
dizio[0]

{}

In [102]:
dizio

defaultdict(dict, {0: {}})

In [103]:
dizio[0][5] = 3

In [104]:
dizio

defaultdict(dict, {0: {5: 3}})

#### Prova costruzione grafo

In [105]:
from collections import defaultdict

frequenze = defaultdict(int)

for cammino in cammini:
        for i in range(len(cammino) - 1):
            u = cammino[i]
            v = cammino[i + 1]
            if u == v:  # ignora self-loop
                continue
            # stampa l'arco
            print(f"Arco: {u} -> {v}")
            u_int, v_int = int(u), int(v)
            arco = (min(u_int, v_int), max(u_int, v_int))
            frequenze[arco] = frequenze.get(arco, 0) + 1
            print(f"Frequenza arco {u} -> {v}: {frequenze[arco]}")


grafo = defaultdict(dict)  # grafo[u][v] = frequenza

for arco, freq in frequenze.items():
    u, v = arco
    print(u, v, freq)
    grafo[u][v] = freq
    grafo[v][u] = freq  # grafo non orientato

print(grafo)

Arco: 4436 -> 6762
Frequenza arco 4436 -> 6762: 1
Arco: 6762 -> 21826
Frequenza arco 6762 -> 21826: 1
Arco: 6939 -> 15290
Frequenza arco 6939 -> 15290: 1
Arco: 15290 -> 2671
Frequenza arco 15290 -> 2671: 1
Arco: 2671 -> 2669
Frequenza arco 2671 -> 2669: 1
Arco: 4826 -> 6939
Frequenza arco 4826 -> 6939: 1
Arco: 6939 -> 3549
Frequenza arco 6939 -> 3549: 1
Arco: 3549 -> 7922
Frequenza arco 3549 -> 7922: 1
Arco: 7922 -> 20214
Frequenza arco 7922 -> 20214: 1
Arco: 14361 -> 15290
Frequenza arco 14361 -> 15290: 1
Arco: 15290 -> 23373
Frequenza arco 15290 -> 23373: 1
Arco: 14361 -> 4766
Frequenza arco 14361 -> 4766: 1
Arco: 4766 -> 1237
Frequenza arco 4766 -> 1237: 1
Arco: 4589 -> 3356
Frequenza arco 4589 -> 3356: 1
Arco: 3356 -> 12956
Frequenza arco 3356 -> 12956: 1
Arco: 12956 -> 3816
Frequenza arco 12956 -> 3816: 1
Arco: 3816 -> 8163
Frequenza arco 3816 -> 8163: 1
Arco: 7575 -> 6939
Frequenza arco 7575 -> 6939: 1
Arco: 6939 -> 22637
Frequenza arco 6939 -> 22637: 1
Arco: 4436 -> 701
Frequenz

In [106]:
dizio

defaultdict(dict, {0: {5: 3}})

In [107]:
dizio[6] = {}

In [108]:
dizio[7][6] = 4

In [109]:
dizio

defaultdict(dict, {0: {5: 3}, 6: {}, 7: {6: 4}})

In [110]:
def remove_node(self, node):
        if node not in self:
            raise ValueError(
                f"Node {node} does not exist in the graph."
            )

        for neighbors in self.values():
            neighbors.pop(node, None)

        del self[node]

In [111]:
remove_node(dizio, 6)

In [112]:
dizio

defaultdict(dict, {0: {5: 3}, 7: {}})

In [113]:
class Graph:

    def __init__(self, directed=False):

        self.adjacency_list = {}  # il grafo sarà un dizionario di dizionari: {nodo: {vicino: peso}} (non usiamo defaultdict per avere più controllo)
        self.directed = directed  # default è False, quindi il grafo è non orientato

## Converte l'identificatore del nodo AS che è una stringa in un intero.

    def _convert_node(self, node):

        try:
            return int(node)
        except (TypeError, ValueError):
            raise ValueError(f"Node {node} is not a valid integer identifier.")

## Definisce come stampare il grafo in modo leggibile

    def __repr__(self):
        graph_str = ""
        for node, neighbors in self.adjacency_list.items():
            graph_str += f" Node {node}: Neighbors and weights {neighbors} \n"
        return graph_str

## Aggiunge un nodo al grafo. Se il nodo esiste già, solleva un'eccezione.

    def add_node(self, node):
        node = self._convert_node(node)

        if node not in self.adjacency_list:
            self.adjacency_list[node] = {}  # aggiunge il nodo con un dizionario (lista di adiacenza e pesi) vuoto
        else:
            raise ValueError(f"Node {node} already exists in the graph.")

## Rimuove un nodo dal grafo. Se il nodo non esiste, solleva un'eccezione.

    def remove_node(self, node):
        node = self._convert_node(node)

        if node not in self.adjacency_list:
            raise ValueError(f"Node {node} does not exist in the graph.")

        for neighbors in self.adjacency_list.values(): ## rimuove il nodo da tutte le liste di adiacenza dei vicini
            neighbors.pop(node, None)

        del self.adjacency_list[node]

## Aggiunge un arco al grafo. Questo arco può essere arbitrario e non proveniente dai cammini BGP (il suo peso sarà None o specificato arbitrariamente dall'utente)

    def add_edge(self, from_node, to_node, weight=None):
        

        #from_node = self._convert_node(from_node) ##ridondanti, lo fa già add_node
        #to_node = self._convert_node(to_node)

        if from_node == to_node:  # elimina i self-loop
            return

        if from_node not in self.adjacency_list:
            self.add_node(from_node)

        if to_node not in self.adjacency_list:
            self.add_node(to_node)

      
        self.adjacency_list[from_node][to_node] = weight

        if not self.directed: ## aggiunge arco in entrambe le direzioni se è undirected
                self.adjacency_list[to_node][from_node] = weight
    
## Rimuove un arco dal grafo. Se i nodi A e B non esistono o se l'arco stesso non esiste (magari i nodi sì ma non sono collegati) lancia un errore.

    def remove_edge(self, from_node, to_node):

        from_node = self._convert_node(from_node)
        to_node = self._convert_node(to_node)

        if from_node not in self.adjacency_list:
            raise ValueError(
                f"Node {from_node} does not exist in the graph."
            )

        if to_node not in self.adjacency_list:
            raise ValueError(
                f"Node {to_node} does not exist in the graph."
            )

        if to_node not in self.adjacency_list[from_node]:
            raise ValueError(
                f"Edge ({from_node}, {to_node}) does not exist in the graph."
            )

        del self.adjacency_list[from_node][to_node]

        if not self.directed:  ## elimina anche l'arco inverso 
            if from_node in self.adjacency_list.get(to_node, {}): ## fa un check se esiste (non dovrebbe servire teoricamente)
                del self.adjacency_list[to_node][from_node]


## Aggiorna la frequenza degli archi

    def update_frequency(self, from_node, to_node):

        from_node = self._convert_node(from_node)
        to_node = self._convert_node(to_node)

        if from_node == to_node:  # elimina i self-loop
            return
        
        ## ho il dubbio che non sia giusto crearli, vediamo

        if from_node not in self.adjacency_list:
            self.add_node(from_node)

        if to_node not in self.adjacency_list:
            self.add_node(to_node)

        #legge la frequenza attuale con .get(...); se l’arco ancora non esiste restituisce 0 come frequenza iniziale e poi aggiunge 1 
        # assegna il nuovo valore a self.adjacency_list[from_node][to_node].

        self.adjacency_list[from_node][to_node] = (                      
            self.adjacency_list[from_node].get(to_node, 0) + 1
        )

        if not self.directed:
            self.adjacency_list[to_node][from_node] = (
                self.adjacency_list[to_node].get(from_node, 0) + 1
            )
    
    def get_neighbors(self, node):
        node = self._convert_node(node)

        if node in self.adjacency_list:
            return self.adjacency_list[node]
        else:
            raise ValueError(f"Node {node} does not exist in the graph.")

    def has_node(self, node):
        node = self._convert_node(node)
        return node in self.adjacency_list

    def has_edge(self, from_node, to_node):
        from_node = self._convert_node(from_node)
        to_node = self._convert_node(to_node)

        if from_node in self.adjacency_list:
            return to_node in self.adjacency_list[from_node]

        return False

    def get_nodes(self):
        return list(self.adjacency_list.keys())

    def get_edges(self):
        edges = []
        seen = set()

        for from_node, neighbors in self.adjacency_list.items():
            for to_node, weight in neighbors.items():

                if not self.directed:
                    arco = (min(from_node, to_node), max(from_node, to_node)) ## prende una unica entry tra (2,3) (3,2)

                    if arco not in seen:
                        seen.add(arco)
                        edges.append((from_node, to_node, weight))

        return edges

    def delete_consecutive_duplicates(self, path): # per esempio path = [10, 10, 20] , conserva solo il primo 10 e 20
        return [
            path[i]
            for i in range(len(path))
            if i == 0 or path[i] != path[i - 1]
        ]

    def add_bgp_path(self, path):
        # converte tutti gli identificatori AS in interi
        path = [self._convert_node(node) for node in path]

        path = self.delete_consecutive_duplicates(path)

        for i in range(len(path) - 1):
            u = path[i]
            v = path[i + 1]

            if u == v:  # elimina i self-loop
                continue

            if not self.has_node(u):
                self.add_node(u)

            if not self.has_node(v):
                self.add_node(v)

            self.update_frequency(u, v)
    

    def largest_connected_component(self):  # Cerchiamo la componente connessa più grande tramite una DFS iterativa.

        visited = set() # nodi già visti

        largest_component = set() # insieme che conterrà i nodi della componente connessa più grande trovata fino a questo momento

        for start_node in self.adjacency_list:

            if start_node in visited:
                continue
                # Saltiamo il resto dell'iterazione e passiamo al nodo successivo se é stato già visto

            component = set() # insieme che conterrà i nodi della componente che stiamo esplorando in questo momento.

            stack = [start_node]
            # Pila utilizzata per eseguire la DFS.
            # Inizialmente contiene solo il nodo di partenza.

            visited.add(start_node)

            while stack:
                # Continuiamo la visita finché ci sono nodi nella pila.

                node = stack.pop()
                # Estraiamo l'ultimo nodo inserito nella pila.
                # Questo comportamento LIFO realizza una DFS.

                component.add(node)
                # Aggiungiamo il nodo alla componente connessa corrente.

                for neighbor in self.adjacency_list[node]:
                    # Scorriamo tutti i vicini del nodo.
                    # Il dizionario interno ha la forma:
                    # {vicino: peso}
                    # Qui vengono considerate solo le chiavi, cioè i vicini.
                    # I pesi non servono per trovare le componenti connesse.

                    if neighbor not in visited:
                        # Consideriamo solo i vicini
                        # che non sono ancora stati visitati.

                        visited.add(neighbor)
                        # Segniamo il vicino come visitato.

                        stack.append(neighbor)
                        # Inseriamo il vicino nella pila,
                        # così verrà esplorato successivamente.

            if len(component) > len(largest_component):
                # Quando la DFS termina, abbiamo trovato
                # un'intera componente connessa.
                # Confrontiamo il suo numero di nodi
                # con quello della componente più grande trovata finora.

                largest_component = component
                # Se la componente corrente è più grande,
                # la salviamo come nuova componente più grande.

        return largest_component
        # Restituiamo l'insieme dei nodi appartenenti
        # alla componente connessa più grande.


    def get_largest_connected_subgraph(self):
        # Troviamo i nodi appartenenti
        # alla componente connessa più grande.

        component_nodes = self.largest_connected_component()

        # Creiamo un nuovo oggetto Graph.
        # Il nuovo grafo mantiene la stessa proprietà del grafo originale:
        # orientato se self.directed è True,
        # non orientato se self.directed è False.

        subgraph = Graph(directed=self.directed)

        for node in component_nodes:
            # Scorriamo tutti i nodi della componente più grande.

            subgraph.add_node(node)
            # Aggiungiamo ogni nodo al nuovo sottografo.
            # In questa fase ogni nodo viene creato
            # con un dizionario dei vicini inizialmente vuoto.

        for from_node in component_nodes:
            # Scorriamo nuovamente tutti i nodi
            # della componente connessa più grande.

            for to_node, weight in self.adjacency_list[from_node].items():
                # Per ogni nodo, scorriamo tutti i suoi vicini
                # e i relativi pesi nel grafo originale.

                if to_node in component_nodes:
                    # Copiamo l'arco soltanto se anche il vicino
                    # appartiene alla componente connessa più grande.

                    subgraph.adjacency_list[from_node][to_node] = weight
                    # Copiamo direttamente l'arco e il suo peso.
                    # Il peso non viene modificato né ricalcolato:
                    # resta uguale a quello presente nel grafo originale.

        return subgraph
        # Restituiamo un nuovo oggetto Graph contenente soltanto
        # la componente connessa più grande.
        # Il grafo originale non viene modificato.



In [114]:
grafo = Graph(directed=False)
for cammino in cammini:
 grafo.add_bgp_path(cammino)

In [115]:
largest_graph = grafo.get_largest_connected_subgraph()

In [116]:
largest_graph

 Node 15290: Neighbors and weights {6939: 1, 2671: 1, 14361: 1, 23373: 1} 
 Node 20485: Neighbors and weights {1273: 1, 21127: 1} 
 Node 21127: Neighbors and weights {20485: 1} 
 Node 4766: Neighbors and weights {14361: 1, 1237: 1} 
 Node 22637: Neighbors and weights {6939: 1} 
 Node 23373: Neighbors and weights {15290: 1} 
 Node 2671: Neighbors and weights {15290: 1, 2669: 1} 
 Node 2669: Neighbors and weights {2671: 1} 
 Node 7922: Neighbors and weights {3549: 1, 20214: 1} 
 Node 1237: Neighbors and weights {4766: 1} 
 Node 20214: Neighbors and weights {7922: 1} 
 Node 7575: Neighbors and weights {6939: 1, 1273: 1} 
 Node 1273: Neighbors and weights {7575: 1, 20485: 1} 
 Node 4826: Neighbors and weights {6939: 1} 
 Node 6939: Neighbors and weights {15290: 1, 4826: 1, 3549: 1, 7575: 1, 22637: 1} 
 Node 3549: Neighbors and weights {6939: 1, 7922: 1} 
 Node 14361: Neighbors and weights {15290: 1, 4766: 1} 

In [117]:
u = 4826
v = 6939

## per verificare che siano presenti entrambi i collegamenti nelle liste con lo stesso peso

print(grafo.adjacency_list[u][v])  # peso da u a v
print(grafo.adjacency_list[v][u])  # peso da v a u

1
1


In [118]:
import pickle
def load_paths(filepath_bz2=None, filepath_pkl=None, max_paths=None):
        
        if filepath_bz2:
            # legge direttamente dal bz2 fermandosi a max_paths
            cammini = []
            contatore = 0
            with bz2.open(filepath_bz2, "rt") as f:
                for riga in f:
                    if contatore >= max_paths:
                        break
                    if riga.startswith("#"):
                        continue
                    parti = riga.strip().split()
                    cammino = []
                    for p in parti[1:]:
                        if "/" in p or "." in p or ":" in p:
                            break
                        nodi = p.split("|")
                        cammino.extend(nodi)
                    if len(cammino) > 1:
                        cammini.append(cammino)
                        contatore += 1
            print(f"cammini letti: {contatore}")
            return cammini
        else:
            # carica tutto dal pickle
            with open(filepath_pkl, "rb") as f:
                return pickle.load(f)

In [119]:
#paths = load_paths(filepath_pkl="/code/ADSproject/data/cammini.pkl")

In [120]:
#len(paths)

In [121]:
import os
import pickle
import argparse
import bz2
import time

class Graph:

    def __init__(self, directed=False):

        self.adjacency_list = {}  # il grafo sarà un dizionario di dizionari: {nodo: {vicino: peso}} (non usiamo defaultdict per avere più controllo)
        self.directed = directed  # default è False, quindi il grafo è non orientato

## Converte l'identificatore del nodo AS che è una stringa in un intero.

    def _convert_node(self, node):

        try:
            return int(node)
        except (TypeError, ValueError):
            raise ValueError(f"Node {node} is not a valid integer identifier.")

## Definisce come stampare il grafo in modo leggibile

    def __repr__(self):
        graph_str = ""
        for node, neighbors in self.adjacency_list.items():
            graph_str += f" Node {node}: Neighbors and weights {neighbors} \n"
        return graph_str

## Aggiunge un nodo al grafo. Se il nodo esiste già, solleva un'eccezione.

    def add_node(self, node):
        node = self._convert_node(node)

        if node not in self.adjacency_list:
            self.adjacency_list[node] = {}  # aggiunge il nodo con un dizionario (lista di adiacenza e pesi) vuoto
        else:
            raise ValueError(f"Node {node} already exists in the graph.")

## Rimuove un nodo dal grafo. Se il nodo non esiste, solleva un'eccezione.

    def remove_node(self, node):
        node = self._convert_node(node)

        if node not in self.adjacency_list:
            raise ValueError(f"Node {node} does not exist in the graph.")

        for neighbors in self.adjacency_list.values(): ## rimuove il nodo da tutte le liste di adiacenza dei vicini
            neighbors.pop(node, None)

        del self.adjacency_list[node]

## Aggiunge un arco al grafo. Questo arco può essere arbitrario e non proveniente dai cammini BGP (il suo peso sarà None o specificato arbitrariamente dall'utente)

    def add_edge(self, from_node, to_node, weight=None):
        

        #from_node = self._convert_node(from_node) ##ridondanti, lo fa già add_node
        #to_node = self._convert_node(to_node)

        if from_node == to_node:  # elimina i self-loop
            return

        if from_node not in self.adjacency_list:
            self.add_node(from_node)

        if to_node not in self.adjacency_list:
            self.add_node(to_node)

      
        self.adjacency_list[from_node][to_node] = weight

        if not self.directed: ## aggiunge arco in entrambe le direzioni se è undirected
                self.adjacency_list[to_node][from_node] = weight
    
## Rimuove un arco dal grafo. Se i nodi A e B non esistono o se l'arco stesso non esiste (magari i nodi sì ma non sono collegati) lancia un errore.

    def remove_edge(self, from_node, to_node):

        from_node = self._convert_node(from_node)
        to_node = self._convert_node(to_node)

        if from_node not in self.adjacency_list:
            raise ValueError(
                f"Node {from_node} does not exist in the graph."
            )

        if to_node not in self.adjacency_list:
            raise ValueError(
                f"Node {to_node} does not exist in the graph."
            )

        if to_node not in self.adjacency_list[from_node]:
            raise ValueError(
                f"Edge ({from_node}, {to_node}) does not exist in the graph."
            )

        del self.adjacency_list[from_node][to_node]

        if not self.directed:  ## elimina anche l'arco inverso 
            if from_node in self.adjacency_list.get(to_node, {}): ## fa un check se esiste (non dovrebbe servire teoricamente)
                del self.adjacency_list[to_node][from_node]


## Aggiorna la frequenza degli archi

    def update_frequency(self, from_node, to_node):

        from_node = self._convert_node(from_node)
        to_node = self._convert_node(to_node)

        if from_node == to_node:  # elimina i self-loop
            return
        
        ## ho il dubbio che non sia giusto crearli, vediamo

        if from_node not in self.adjacency_list:
            self.add_node(from_node)

        if to_node not in self.adjacency_list:
            self.add_node(to_node)

        #legge la frequenza attuale con .get(...); se l’arco ancora non esiste restituisce 0 come frequenza iniziale e poi aggiunge 1 
        # assegna il nuovo valore a self.adjacency_list[from_node][to_node].

        self.adjacency_list[from_node][to_node] = (                      
            self.adjacency_list[from_node].get(to_node, 0) + 1
        )

        if not self.directed:
            self.adjacency_list[to_node][from_node] = (
                self.adjacency_list[to_node].get(from_node, 0) + 1
            )
    
    def get_neighbors(self, node):
        node = self._convert_node(node)

        if node in self.adjacency_list:
            return self.adjacency_list[node]
        else:
            raise ValueError(f"Node {node} does not exist in the graph.")

    def has_node(self, node):
        node = self._convert_node(node)
        return node in self.adjacency_list

    def has_edge(self, from_node, to_node):
        from_node = self._convert_node(from_node)
        to_node = self._convert_node(to_node)

        if from_node in self.adjacency_list:
            return to_node in self.adjacency_list[from_node]

        return False

    def get_nodes(self):
        return list(self.adjacency_list.keys())

    def get_edges(self):
        edges = []
        seen = set()

        for from_node, neighbors in self.adjacency_list.items():
            for to_node, weight in neighbors.items():

                if not self.directed:
                    arco = (min(from_node, to_node), max(from_node, to_node)) ## prende una unica entry tra (2,3) (3,2)

                    if arco not in seen:
                        seen.add(arco)
                        edges.append((from_node, to_node, weight))

        return edges

    def delete_consecutive_duplicates(self, path): # per esempio path = [10, 10, 20] , conserva solo il primo 10 e 20
        return [
            path[i]
            for i in range(len(path))
            if i == 0 or path[i] != path[i - 1]
        ]

    def add_bgp_path(self, path):
        # converte tutti gli identificatori AS in interi
        path = [self._convert_node(node) for node in path]

        path = self.delete_consecutive_duplicates(path)

        for i in range(len(path) - 1):
            u = path[i]
            v = path[i + 1]

            if u == v:  # elimina i self-loop
                continue

            if not self.has_node(u):
                self.add_node(u)

            if not self.has_node(v):
                self.add_node(v)

            self.update_frequency(u, v)
    

    def largest_connected_component(self):  # Cerchiamo la componente connessa più grande tramite una DFS iterativa.

        visited = set() # nodi già visti

        largest_component = set() # insieme che conterrà i nodi della componente connessa più grande trovata fino a questo momento

        for start_node in self.adjacency_list:

            if start_node in visited:
                continue
                # Saltiamo il resto dell'iterazione e passiamo al nodo successivo se é stato già visto

            component = set() # insieme che conterrà i nodi della componente che stiamo esplorando in questo momento.

            stack = [start_node]
            # Pila utilizzata per eseguire la DFS.
            # Inizialmente contiene solo il nodo di partenza.

            visited.add(start_node)

            while stack:
                # Continuiamo la visita finché ci sono nodi nella pila.

                node = stack.pop()
                # Estraiamo l'ultimo nodo inserito nella pila.
                # Questo comportamento LIFO realizza una DFS.

                component.add(node)
                # Aggiungiamo il nodo alla componente connessa corrente.

                for neighbor in self.adjacency_list[node]:
                    # Scorriamo tutti i vicini del nodo.
                    # Il dizionario interno ha la forma:
                    # {vicino: peso}
                    # Qui vengono considerate solo le chiavi, cioè i vicini.
                    # I pesi non servono per trovare le componenti connesse.

                    if neighbor not in visited:
                        # Consideriamo solo i vicini
                        # che non sono ancora stati visitati.

                        visited.add(neighbor)
                        # Segniamo il vicino come visitato.

                        stack.append(neighbor)
                        # Inseriamo il vicino nella pila,
                        # così verrà esplorato successivamente.

            if len(component) > len(largest_component):
                # Quando la DFS termina, abbiamo trovato
                # un'intera componente connessa.
                # Confrontiamo il suo numero di nodi
                # con quello della componente più grande trovata finora.

                largest_component = component
                # Se la componente corrente è più grande,
                # la salviamo come nuova componente più grande.

        return largest_component
        # Restituiamo l'insieme dei nodi appartenenti
        # alla componente connessa più grande.


    def get_largest_connected_subgraph(self):
        # Troviamo i nodi appartenenti
        # alla componente connessa più grande.

        component_nodes = self.largest_connected_component()

        # Creiamo un nuovo oggetto Graph.
        # Il nuovo grafo mantiene la stessa proprietà del grafo originale:
        # orientato se self.directed è True,
        # non orientato se self.directed è False.

        subgraph = Graph(directed=self.directed)

        for node in component_nodes:
            # Scorriamo tutti i nodi della componente più grande.

            subgraph.add_node(node)
            # Aggiungiamo ogni nodo al nuovo sottografo.
            # In questa fase ogni nodo viene creato
            # con un dizionario dei vicini inizialmente vuoto.

        for from_node in component_nodes:
            # Scorriamo nuovamente tutti i nodi
            # della componente connessa più grande.

            for to_node, weight in self.adjacency_list[from_node].items():
                # Per ogni nodo, scorriamo tutti i suoi vicini
                # e i relativi pesi nel grafo originale.

                if to_node in component_nodes:
                    # Copiamo l'arco soltanto se anche il vicino
                    # appartiene alla componente connessa più grande.

                    subgraph.adjacency_list[from_node][to_node] = weight
                    # Copiamo direttamente l'arco e il suo peso.
                    # Il peso non viene modificato né ricalcolato:
                    # resta uguale a quello presente nel grafo originale.

        return subgraph
        # Restituiamo un nuovo oggetto Graph contenente soltanto
        # la componente connessa più grande.
        # Il grafo originale non viene modificato.

    @staticmethod
    def load_paths(filepath_bz2=None, filepath_pkl=None, max_paths=None):
        
        if filepath_bz2:
            # legge direttamente dal bz2 fermandosi a max_paths
            cammini = []
            contatore = 0
            with bz2.open(filepath_bz2, "rt") as f:
                for riga in f:
                    if contatore >= max_paths:
                        break
                    if riga.startswith("#"):
                        continue
                    parti = riga.strip().split()
                    cammino = []
                    for p in parti[1:]:
                        if "/" in p or "." in p or ":" in p:
                            break
                        nodi = p.split("|")
                        cammino.extend(nodi)
                    if len(cammino) > 1:
                        cammini.append(cammino)
                        contatore += 1
            print(f"cammini letti: {contatore}")
            return cammini
        else:
            # carica tutto dal pickle
            with open(filepath_pkl, "rb") as f:
                return pickle.load(f)

    def build_from_bz2(self, filepath, max_paths=None):

        contatore = 0
        with bz2.open(filepath, "rt") as f:
            for riga in f:
                if max_paths and contatore >= max_paths:
                    break
                if riga.startswith("#"):
                    continue
                parti = riga.strip().split()
                cammino = []
                for p in parti[1:]:
                    if "/" in p or "." in p or ":" in p:
                        break
                    nodi = p.split("|")
                    cammino.extend(nodi)
                if len(cammino) > 1:
                    self.add_bgp_path(cammino)
                    contatore += 1
        print(f"cammini letti: {contatore}")

    def build_from_paths(self, paths):
        for path in paths:
            self.add_bgp_path(path)

    def save_graph(self, filepath):

        with open(filepath, "wb") as f:
            pickle.dump(self, f)

    @staticmethod
    def load_graph(filepath):

        with open(filepath, "rb") as f:
            return pickle.load(f)

In [122]:
import pickle
import time

inizio = time.time()
with open("/code/ADSproject/data/cammini_test.pkl", "rb") as f:
    cammini = pickle.load(f)
fine = time.time()
print(f"tempo caricamento pkl: {fine - inizio:.2f} secondi")

inizio = time.time()
grafo = Graph(directed=False)
grafo.build_from_paths(cammini)  # limita a 1M
fine = time.time()
print(f"tempo costruzione grafo da pkl: {fine - inizio:.2f} secondi")

tempo caricamento pkl: 0.59 secondi
tempo costruzione grafo da pkl: 4.10 secondi


In [123]:
len(grafo.get_edges())


65910

In [124]:
len(grafo.get_nodes())

37020

### MST and Union-Find Understanding

In [125]:
## Union-Find senza ottimizzazioni

class UnionFind:
    def __init__(self,n):
        self.parent = list(range(n)) ## inizializza una lista n -1 (elementi o nodi), ogni elemento è il suo stesso parent (root)

    def find (self,x):
        if(self.parent[x] != x): ## se il nodo non è parent di se stesso 
            return self.find(self.parent[x]) ## allora trovami il parent 

        return self.parent[x] # ritorna il parent 

    def union(self,a,b):
        rootA = self.find(a) 
        rootB = self.find(b)

        if (rootA != rootB): # se appartengono a due set diversi, unisci 
            self.parent[rootB] = rootA # A diventa la sua radice

uf = UnionFind(5)
uf.union(0,1)
uf.union(2,3)
uf.union(0,2)
print(uf.find(0),uf.find(1))
print(uf.find(2),uf.find(3))
print(uf.find(0),uf.find(2))

0 0
0 0
0 0


In [126]:
## Union-Find ottimizzata con:
## - path compression: anzichè formare una fila lunghissima di nodi attaccati a quella radice (per cui poi per trovarlo devo andare a ritroso tantissimo) 
## attaco direttamente i nuovi elementi alla radice 
class UnionFind:
    def __init__(self, n):
        self.parent = [i for i in range(n)]
        self.rank = [0] * n # inizializza a 0 il rank (altezza) di ogni nodo

    def find(self, x):
        if self.parent[x] != x:  # se x non è la root
            # Path compression: risale ricorsivamente fino alla root e tornando indietro collega direttamente tutti i nodi incontrati lungo il percorso
            self.parent[x] = self.find(self.parent[x])

        return self.parent[x]

    def union(self, x, y):
        rootX = self.find(x)
        rootY = self.find(y)

        # Union by rank 
        if rootX != rootY: # se le root sono diverse (set diversi)
            if self.rank[rootX] < self.rank[rootY]: #se il rango è minore
                self.parent[rootX] = rootY #assegna la root col rango maggiore
            elif self.rank[rootX] > self.rank[rootY]: # viceversa
                self.parent[rootY] = rootX
            else:
                self.parent[rootY] = rootX #se sono uguali incrementa il rango
                self.rank[rootX] += 1
            return True # unione eseguita
        return False # erano nello stesso set

In [127]:
uf = UnionFind(5)

print("Stato iniziale")
print("parent:", uf.parent)
print("rank:  ", uf.rank)

assert uf.union(0, 1) is True
print("\nDopo union(0, 1)")
print("parent:", uf.parent)
print("rank:  ", uf.rank)

assert uf.union(2, 3) is True
print("\nDopo union(2, 3)")
print("parent:", uf.parent)
print("rank:  ", uf.rank)

assert uf.union(0, 2) is True
print("\nDopo union(0, 2)")
print("parent:", uf.parent)
print("rank:  ", uf.rank)

# 0, 1, 2 e 3 devono essere nella stessa componente.
assert uf.find(0) == uf.find(1)
assert uf.find(0) == uf.find(2)
assert uf.find(0) == uf.find(3)

# 4 deve essere ancora separato.
assert uf.find(0) != uf.find(4)

# L'unione non deve avvenire perché 1 e 3
# sono già nella stessa componente.
assert uf.union(1, 3) is False

print("\nStato dopo le find e la path compression")
print("parent:", uf.parent)
print("rank:  ", uf.rank)

print("\nRoot dei nodi:")
for node in range(5):
    print(f"find({node}) = {uf.find(node)}")

Stato iniziale
parent: [0, 1, 2, 3, 4]
rank:   [0, 0, 0, 0, 0]

Dopo union(0, 1)
parent: [0, 0, 2, 3, 4]
rank:   [1, 0, 0, 0, 0]

Dopo union(2, 3)
parent: [0, 0, 2, 2, 4]
rank:   [1, 0, 1, 0, 0]

Dopo union(0, 2)
parent: [0, 0, 0, 2, 4]
rank:   [2, 0, 1, 0, 0]

Stato dopo le find e la path compression
parent: [0, 0, 0, 0, 4]
rank:   [2, 0, 1, 0, 0]

Root dei nodi:
find(0) = 0
find(1) = 0
find(2) = 0
find(3) = 0
find(4) = 4


In [128]:
def kruskal(n, edges):
        edges.sort(key=lambda x: x[2]) # ordina per peso/frequenza gli archi (che in questo esempio sono tuple)

        ds = UnionFind(n)
        mst_weight = 0 # peso iniziale del mst
        mst_edges = []

        for u, v, w in edges: # per vertici e peso
            if ds.union(u, v): # se viene performata l'unione
                mst_weight += w # allora aumenta il costo totale del mst
                mst_edges.append((u, v, w)) #ricostrusci mst

        return mst_weight, mst_edges

In [129]:
edges = [
    (0, 1, 10),
    (0, 2, 6),
    (0, 3, 5),
    (1, 3, 15),
    (2, 3, 4)
]
n = 4 


mst_weight, mst_edges = kruskal(n, edges)

print("Edges in the Minimum Spanning Tree:")
for u, v, w in mst_edges:
    print(f"{u} -- {v} == {w}")

print("Total Weight of MST:", mst_weight)

Edges in the Minimum Spanning Tree:
2 -- 3 == 4
0 -- 3 == 5
0 -- 1 == 10
Total Weight of MST: 19


In [130]:
def kruskal(graph):
    if graph.directed:
        raise ValueError(
            "Kruskal richiede un grafo non orientato."
        )

    nodes = graph.get_nodes()
    n = len(nodes)

    if n == 0:
        return {}, 0

    # Gli AS number non sono necessariamente 0, 1, 2, ...
    # Li associamo temporaneamente a indici consecutivi,
    # così UnionFind può usare liste.
    node_to_index = {
        node: index
        for index, node in enumerate(nodes)
    }

    # Ogni arco ha la forma:
    # (from_node, to_node, frequenza)
    edges = graph.get_edges()

    for u, v, weight in edges:
        if weight is None:
            raise ValueError(
                f"L'arco ({u}, {v}) non ha una frequenza valida."
            )

    # Ordine crescente di frequenza.
    # Serve per costruire un Minimum Spanning Tree.
    edges.sort(key=lambda edge: edge[2])

    union_find = UnionFind(n)

    # Rappresentazione dell'MST come lista di adiacenza:
    # {nodo: [(vicino, peso), ...]}
    mst = {
        node: []
        for node in nodes
    }

    mst_weight = 0
    selected_edges = 0

    for u, v, weight in edges:
        index_u = node_to_index[u]
        index_v = node_to_index[v]

        # True significa che u e v erano in componenti diverse.
        if union_find.union(index_u, index_v):
            mst[u].append((v, weight))
            mst[v].append((u, weight))

            mst_weight += weight
            selected_edges += 1

            # Un albero con n nodi contiene esattamente n - 1 archi.
            if selected_edges == n - 1:
                break

    # Se il grafo passato è connesso, devono esserci n - 1 archi.
    if selected_edges != n - 1:
        raise ValueError(
            "Il grafo non è connesso. "
            "Usare prima la componente connessa più grande."
        )

    return mst, mst_weight

In [131]:
# Nel tuo codice stai già estraendo la componente maggiore:
largest_graph = grafo.get_largest_connected_subgraph()

mst, mst_weight = kruskal(largest_graph)

print("Peso totale MST:", mst_weight)
print("Numero di nodi:", len(mst))
print(
    "Numero di archi MST:",
    sum(len(neighbors) for neighbors in mst.values()) // 2
)

Peso totale MST: 273951
Numero di nodi: 37020
Numero di archi MST: 37019


In [132]:
def minimax_query_dfs(mst, start, target):
    if start not in mst:
        raise ValueError(f"Il nodo {start} non esiste nell'MST.")

    if target not in mst:
        raise ValueError(f"Il nodo {target} non esiste nell'MST.")

    if start == target:
        return 0

    visited = {start}

    # Ogni elemento contiene:
    # (nodo corrente, massimo peso incontrato dal nodo iniziale)
    stack = [(start, 0)]

    while stack:
        node, current_max = stack.pop()

        if node == target:
            return current_max

        for neighbor, weight in mst[node]:
            if neighbor not in visited:
                visited.add(neighbor)

                new_max = max(current_max, weight)

                stack.append((neighbor, new_max))

    raise ValueError(
        f"Non esiste un cammino tra {start} e {target}."
    )

In [134]:
grafo

 Node 4436: Neighbors and weights {6762: 4005, 701: 8889, 2914: 30307, 3549: 12411, 4134: 758, 23352: 572, 209: 3917, 6461: 1863, 6939: 3743, 1785: 504, 3491: 3139, 4323: 3036, 1273: 4974, 6830: 433, 5400: 313, 22637: 59, 7473: 3916, 286: 796, 4766: 167, 22773: 774, 22212: 663, 22561: 687, 10026: 714, 40015: 118, 5650: 447, 15412: 1681, 15290: 216, 3786: 1203, 4837: 1041, 4637: 893, 25653: 52, 15003: 289, 852: 305, 2497: 1016, 9002: 833, 4589: 691, 7843: 1123, 19151: 967, 3303: 246, 11164: 597, 32475: 64, 20473: 236, 577: 880, 4565: 502, 22822: 117, 4725: 372, 20115: 543, 12041: 69, 11814: 317, 6079: 49, 10929: 35, 1257: 195, 2119: 126, 3595: 76, 8218: 268, 13237: 168, 6128: 194, 7132: 217, 16702: 11, 26972: 23, 6327: 261, 6315: 29, 15557: 115, 5769: 102, 11074: 19, 15169: 44, 8121: 137, 19029: 83, 8001: 113, 8928: 112, 19397: 57, 14390: 81, 6539: 130, 16626: 80, 4788: 75, 11758: 160, 7065: 182, 20940: 213, 3292: 287, 46562: 58, 27506: 194, 32181: 23, 32748: 98, 11537: 43, 5413: 18, 22

In [137]:
largest_graph = grafo.get_largest_connected_subgraph()

mst, mst_weight = kruskal(largest_graph)

u = 7992
v = 15290

cost = minimax_query_dfs(mst, u, v)

print(f"Costo minimax tra {u} e {v}: {cost}")

Costo minimax tra 7992 e 15290: 1


In [139]:
from collections import Counter

edges = grafo.get_edges()

frequencies = Counter(weight for _, _, weight in edges)

print("Frequenze più comuni:")
for weight, count in frequencies.most_common(10):
    print(f"peso {weight}: {count} archi")

print("Archi totali:", len(edges))
print("Archi con peso 1:", frequencies[1])

Frequenze più comuni:
peso 1: 10098 archi
peso 2: 9347 archi
peso 3: 8170 archi
peso 4: 5857 archi
peso 5: 4228 archi
peso 6: 3039 archi
peso 7: 2336 archi
peso 8: 1890 archi
peso 9: 1548 archi
peso 10: 1298 archi
Archi totali: 65910
Archi con peso 1: 10098


In [140]:
mst, mst_weight = kruskal(grafo)

mst_frequencies = []

for node, neighbors in mst.items():
    for neighbor, weight in neighbors:
        if node < neighbor:
            mst_frequencies.append(weight)

print("Pesi distinti nell'MST:", sorted(set(mst_frequencies)))
print("Peso minimo:", min(mst_frequencies))
print("Peso massimo:", max(mst_frequencies))

print("Distribuzione MST:")
for weight, count in Counter(mst_frequencies).most_common():
    print(f"peso {weight}: {count} archi")

Pesi distinti nell'MST: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 140, 142, 146, 147, 148, 150, 151, 153, 156, 159, 160, 161, 163, 164, 165, 166, 168, 169, 171, 176, 178, 184, 185, 186, 187, 188, 190, 192, 195, 196, 197, 198, 201, 203, 205, 210, 211, 213, 217, 218, 225, 226, 231, 233, 234, 235, 238, 239, 242, 250, 251, 255, 258, 263, 264, 268, 279, 281, 285, 287, 288, 289, 294, 302, 304, 313, 314, 315, 322, 330, 332, 341, 345, 356, 373, 376, 379, 382, 391

In [138]:
mst_test = {
    0: [(1, 1)],
    1: [(0, 1), (2, 4)],
    2: [(1, 4), (3, 2)],
    3: [(2, 2)]
}

print(minimax_query_dfs(mst_test, 0, 3))

4


In [141]:
def minimax_query_dfs(mst, start, target):
    if start not in mst or target not in mst:
        raise ValueError("Nodo non presente nell'MST")

    if start == target:
        return 0, [start]

    visited = {start}
    stack = [(start, 0, [start])]

    while stack:
        node, current_max, path = stack.pop()

        if node == target:
            return current_max, path

        for neighbor, weight in mst[node]:
            if neighbor not in visited:
                visited.add(neighbor)

                stack.append((
                    neighbor,
                    max(current_max, weight),
                    path + [neighbor]
                ))

    raise ValueError("Cammino non trovato")

In [142]:
cost, path = minimax_query_dfs(mst, u, v)

print("Cammino:", path)
print("Costo minimax:", cost)

print("Archi attraversati:")
for a, b in zip(path, path[1:]):
    for neighbor, weight in mst[a]:
        if neighbor == b:
            print(a, "--", b, "peso:", weight)
            break

Cammino: [7992, 35843, 3356, 24851, 6939, 40630, 3549, 31980, 174, 36273, 15290]
Costo minimax: 1
Archi attraversati:
7992 -- 35843 peso: 1
35843 -- 3356 peso: 1
3356 -- 24851 peso: 1
24851 -- 6939 peso: 1
6939 -- 40630 peso: 1
40630 -- 3549 peso: 1
3549 -- 31980 peso: 1
31980 -- 174 peso: 1
174 -- 36273 peso: 1
36273 -- 15290 peso: 1
